In [ ]:
import os
import matplotlib.pyplot as plt

RESULT_DIR = "/home/atsushi/ros2-picas/results/"
TARGET_LABELS = ['C1R1_12_latency', 'C2R3_11_latency', 'C3R2_7_latency', 'C4R3_4_latency']
PLOT_ORDER_MAP = {
    "Default Executors": [
        'case_study_default_st4',
        'case_study_default_mt4',
        'case_study_default_mt_separate4',
    ],
    "Custom Executors": [
        'case_study_picas_st4',
        'case_study_picas_mt4',
        'case_study_picas_mt_separate4',
        'case_study_cie_4',
    ]
}
X_LABEL_MAP = {
    'case_study_cie_4': 'GFP-CIE',
    'case_study_picas_mt4': 'PiCAS-Multi (1)',
    'case_study_picas_mt_separate4': 'PiCAS-Multi (2)',
    'case_study_picas_st4': 'PiCAS-Single',
    'case_study_default_mt_separate4': 'Multi (2)',
    'case_study_default_mt4': 'Multi (1)',
    'case_study_default_st4': 'Single',
}


def read_rt_data(path, key, combined_data):
    with open(path, 'r') as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) != 3 or parts[2] == '0':
                continue
            label, value = parts[0], int(parts[1])
            if label in combined_data:
                combined_data[label][key].append(value / 1000)  # μs -> ms


# データ読み込みの準備
keys = [dir.split('/')[-1] for dir in os.listdir(RESULT_DIR)]
combined_data = {label: {key: [] for key in keys} for label in TARGET_LABELS}

for dir in os.listdir(RESULT_DIR):
    key = dir.split('/')[-1]
    target_file = os.path.join(RESULT_DIR, dir, 'R.txt')
    read_rt_data(target_file, key, combined_data)

# グラフの描画
for label in TARGET_LABELS:
    default_keys = [key for key in keys if key.startswith("case_study_default_")]
    other_keys = [key for key in keys if not key.startswith("case_study_default_")]

    default_data = [combined_data[label][key] for key in default_keys]
    other_data = [combined_data[label][key] for key in other_keys]

    fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=False)

    for idx, (ax, data_group, group_keys, title, cmap_name) in enumerate(zip(
        axes,
        [default_data, other_data],
        [default_keys, other_keys],
        ["Default Executors", "Custom Executors"],
        ['tab10', 'Set2']  # 異なるカラーマップを指定
    )):
        ordered_keys = PLOT_ORDER_MAP.get(title, group_keys)
        ordered_data = [combined_data[label][key] for key in ordered_keys]

        cmap = plt.get_cmap(cmap_name)
        colors = [cmap(i) for i in range(len(ordered_data))]

        parts = ax.violinplot(ordered_data, showmeans=True, showmedians=True)

        for body, color in zip(parts['bodies'], colors):
            body.set_facecolor(color)
            body.set_edgecolor(color)
            body.set_linewidth(1.5)
            body.set_alpha(0.8)

        for line_key in ['cmedians', 'cmeans', 'cbars', 'cmins', 'cmaxes']:
            if line_key in parts:
                lines = parts[line_key]
                if isinstance(lines, list):  # 一般的には list of Line2D
                    for line, color in zip(lines, colors):
                        line.set_color(color)
                        line.set_linewidth(1.5)
                else:
                    # 例：cbars や cmeans は LineCollection の可能性あり
                    lines.set_color(colors)
                    lines.set_linewidth(1.5)

        ax.set_xticks([i + 1 for i in range(len(ordered_keys))])
        ax.set_xticklabels(
            [X_LABEL_MAP.get(key, key) for key in ordered_keys],
            rotation=30, ha='right'
        )
        ax.set_title(title)
        ax.grid(False)

    axes[0].set_ylabel('Response Time [ms]', fontsize=12)
    fig.suptitle(f'{label[:2].replace("C", "Chain ")}', fontsize=14)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    # plt.show()
    plt.savefig(f'{label}.jpg')